# 02 — Feature engineering: composition + symmetry descriptors
**Project:** MAG2D-NC | **Phase:** F4 / Step 3 | **Protocol:** v1.1 (frozen; permutation-test amendment logged)

Input: `spiral_labels_v1_1.csv` (164 materials, gate-verified, dual-validated).
Output: descriptor matrix for tasks T1/T2 — Magpie composition features +
physics-informed symmetry/SOC-proxy features. Structure-graph features come
later (separate notebook) once per-material structures are available; nothing
here depends on them.

Descriptor groups (frozen ablation axes A1 of protocol §P8):
- `comp_*` : Magpie elemental-property statistics (matminer)
- `soc_*`  : SOC proxies — anion Z statistics (heavy ligands drive anisotropy/DMI)
- `sym_*`  : lattice/symmetry — inversion-symmetry flag (DMI requires broken
             inversion), crystal-system encoding, magnetic-species Z

## CONFIG

In [ ]:
from pathlib import Path
from datetime import datetime

CONFIG = {
    "PROJECT_ROOT": Path.home() / "MAG2D-NC",
    "LABELS_CSV": Path.home() / "MAG2D-NC" / "dataset" / "spiral_labels_v1_1.csv",
    "RUN_STAMP": datetime.now().strftime("%Y%m%d-%H%M%S"),
}
OUT_DIR = CONFIG["PROJECT_ROOT"] / "dataset"
print({k: str(v) for k, v in CONFIG.items()})

## Dependencies
`matminer` pulls in `pymatgen`; first install is a few hundred MB — one-time cost.

In [ ]:
import importlib, subprocess, sys
for pkg, mod in [("pandas","pandas"),("pyarrow","pyarrow"),
                 ("pymatgen","pymatgen"),("matminer","matminer")]:
    try:
        importlib.import_module(mod); print(pkg, "OK")
    except ImportError:
        print(pkg, "installing ...")
        subprocess.run([sys.executable,"-m","pip","install",pkg], check=True)
        print(pkg, "installed")

## Load labels (verification: gate counts must still hold)

In [ ]:
import pandas as pd

df = pd.read_csv(CONFIG["LABELS_CSV"])
assert len(df) == 164, f"Expected 164 rows, got {len(df)}"
c4 = df["label4"].value_counts().to_dict()
assert c4.get("FM")==58 and c4.get("AFM_collinear")==21 \
   and c4.get("NC")==70 and c4.get("DM_SS")==15, f"Gate mismatch: {c4}"
print("Gate re-verified on load:", c4)
print("label2:", df["label2"].value_counts().to_dict())
df.head(3)

## Magpie composition descriptors
Standard 132-feature elemental-property statistics; the field's baseline
representation (Ward et al. 2016), so reviewers can calibrate our numbers.

In [ ]:
from pymatgen.core import Composition
from matminer.featurizers.composition import ElementProperty

df["composition"] = df["formula"].map(Composition)
ep = ElementProperty.from_preset("magpie")
feat = ep.featurize_dataframe(df.copy(), col_id="composition", ignore_errors=False)
magpie_cols = [c for c in feat.columns if c.startswith("MagpieData")]
feat = feat.rename(columns={c: "comp_" + c.replace("MagpieData ", "").replace(" ", "_")
                            for c in magpie_cols})
comp_cols = [c for c in feat.columns if c.startswith("comp_")]
print(f"Magpie features: {len(comp_cols)}")
assert feat[comp_cols].isna().sum().sum() == 0, "NaNs in Magpie features"
print("No NaNs. Example columns:", comp_cols[:5])

## SOC-proxy descriptors (physics-informed)
Anisotropy and DMI in 2D magnets are driven by spin-orbit coupling on the
*ligands* (heavy halides/chalcogens), not just any heavy element. We therefore
compute Z statistics restricted to non-magnetic p-block anions — a deliberate,
interpretable addition beyond generic Magpie Z stats.

In [ ]:
import re
import numpy as np

MAGNETIC = {"Sc","Ti","V","Cr","Mn","Fe","Co","Ni","Cu","Y","Zr","Nb","Mo","Tc",
            "Ru","Rh","Pd","Re","Os","Ir","Pt","Ag","Au","Ta","W","Hf"}
ANION_CANDIDATES = {"F","Cl","Br","I","O","S","Se","Te","N","P","As","H"}

from pymatgen.core.periodic_table import Element

def anion_stats(formula):
    counts = {}
    for el, n in re.findall(r"([A-Z][a-z]?)(\d*)", formula):
        if el: counts[el] = counts.get(el, 0) + (int(n) if n else 1)
    anions = {el: n for el, n in counts.items() if el in ANION_CANDIDATES}
    if not anions:
        return pd.Series({"soc_anion_Z_max": 0.0, "soc_anion_Z_mean": 0.0,
                          "soc_anion_Z_wmean": 0.0, "soc_heavy_anion_frac": 0.0})
    Zs = np.array([Element(el).Z for el in anions])
    ws = np.array(list(anions.values()), dtype=float)
    return pd.Series({
        "soc_anion_Z_max": float(Zs.max()),
        "soc_anion_Z_mean": float(Zs.mean()),
        "soc_anion_Z_wmean": float((Zs*ws).sum()/ws.sum()),
        "soc_heavy_anion_frac": float(ws[Zs >= 34].sum()/ws.sum()),  # Se and heavier
    })

soc = df["formula"].apply(anion_stats)
feat = pd.concat([feat, soc], axis=1)
print(soc.describe().round(2))

## Symmetry descriptors
`sym_has_inversion` matters physically: DMI (and hence chiral spirals) requires
broken inversion symmetry. The lookup below covers exactly the layer groups
present in this dataset and is asserted complete — an unknown group stops the
notebook rather than defaulting.

In [ ]:
# point-group-level inversion symmetry for the layer groups in this dataset
HAS_INVERSION = {
    "P-31m": True,  "P-3": True,   "P312": False, "C2": False,  "P-1": True,
    "P2/m": True,   "P3m1": False, "P-3m1": True, "Pmm2": False,"P1": False,
    "Pmmm": True,   "C2/m": True,  "P-6m2": False,"Cm": False,  "Pm": False,
    "P-4m2": False,
}
unknown = set(df["sg"].unique()) - set(HAS_INVERSION)
assert not unknown, f"Layer groups missing from lookup: {unknown}"

CRYSTAL_SYSTEM = {
    "P-31m":"trigonal","P-3":"trigonal","P312":"trigonal","P3m1":"trigonal",
    "P-3m1":"trigonal","P-6m2":"hexagonal","P-4m2":"tetragonal",
    "Pmm2":"orthorhombic","Pmmm":"orthorhombic","C2":"monoclinic",
    "C2/m":"monoclinic","P2/m":"monoclinic","Cm":"monoclinic","Pm":"monoclinic",
    "P1":"triclinic","P-1":"triclinic",
}
feat["sym_has_inversion"] = feat["sg"].map(HAS_INVERSION).astype(int)
cs = pd.get_dummies(feat["sg"].map(CRYSTAL_SYSTEM), prefix="sym_cs").astype(int)
feat = pd.concat([feat, cs], axis=1)

def mag_Z(formula):
    els = set(re.findall(r"[A-Z][a-z]?", formula)) & MAGNETIC
    return float(np.mean([Element(e).Z for e in els])) if els else 0.0
feat["sym_mag_species_Z"] = feat["formula"].map(mag_Z)

print("inversion flag vs label2 (physics sanity — DM_SS must all be inversion-broken):")
print(pd.crosstab(feat["label4"], feat["sym_has_inversion"]))
assert (feat.loc[feat.label4=="DM_SS","sym_has_inversion"]==0).all(), \
    "A DM_SS material with inversion symmetry would contradict DMI physics — check lookup."
print("Physics sanity check PASSED: all 15 DM_SS lack inversion symmetry.")

## Assemble + save
Feature matrix = comp_* + soc_* + sym_* columns; labels and group_id carried
alongside. Timestamped parquet; nothing overwritten.

In [ ]:
feature_cols = [c for c in feat.columns
                if c.startswith(("comp_","soc_","sym_"))]
keep = ["formula","sg","q_raw","label4","label2","label_src","group_id"] + feature_cols
X = feat[keep].copy()

assert X[feature_cols].isna().sum().sum() == 0, "NaNs in final matrix"
zero_var = [c for c in feature_cols if X[c].nunique() <= 1]
print(f"{len(feature_cols)} features | {len(zero_var)} zero-variance (kept, flagged):")
print(zero_var[:10])

out = OUT_DIR / f"features_T1_{CONFIG['RUN_STAMP']}.parquet"
X.to_parquet(out, index=False)

card = {
    "file": out.name, "n_rows": int(len(X)), "n_features": len(feature_cols),
    "groups": int(X["group_id"].nunique()),
    "feature_groups": {"comp": sum(c.startswith('comp_') for c in feature_cols),
                        "soc": sum(c.startswith('soc_') for c in feature_cols),
                        "sym": sum(c.startswith('sym_') for c in feature_cols)},
    "source_labels": CONFIG["LABELS_CSV"].name,
    "created": CONFIG["RUN_STAMP"],
}
import json as _j
(CONFIG["PROJECT_ROOT"]/"output"/f"datacard_features_{CONFIG['RUN_STAMP']}.json").write_text(_j.dumps(card, indent=2))
print(_j.dumps(card, indent=2))

## Next
Notebook 03: D3/D4 acquisition (JARVIS-2D + 2DMatPedia) for T4/LODO.
Notebook 04 (F5): T1 baselines — dummy, logistic, LightGBM on this matrix,
with the grouped-CV + permutation-test protocol. Pending approval.